# 25 - Polars: operaciones habituales y flujo lazy

En este cuaderno practicamos operaciones intermedias con **Polars**, sin depender de pandas ni de NumPy. Todos los datos se crean en memoria para que el ejemplo sea reproducible.

In [ ]:
import polars as pl

print(f"Polars {pl.__version__}")

## 1. Crear un DataFrame y explorar su estructura

`pl.DataFrame` permite construir una tabla a partir de listas o diccionarios. En este ejemplo dejamos un valor nulo para practicar su tratamiento posterior.

In [ ]:
ventas = pl.DataFrame({
    "venta_id": [1, 2, 3, 4, 5, 6, 6],
    "producto": ["libreta", "bolígrafo", "mochila", "libreta", "mochila", "carpeta", "carpeta"],
    "categoria": ["papelería", "papelería", "equipaje", "papelería", "equipaje", "papelería", "papelería"],
    "precio": [3.5, 1.2, 24.0, 3.5, 24.0, 5.75, 5.75],
    "unidades": [4, 10, 2, 3, None, 6, 6],
    "cliente_id": [101, 102, 101, 103, 102, 104, 104],
})

print(ventas)
print(ventas.schema)

## 2. Selección, alias, expresiones y filtros

Las expresiones (`pl.col`, `pl.lit`, `pl.when`) describen la transformación de cada columna. Se pueden combinar en `select` y `filter`.

In [ ]:
resumen = ventas.select(
    pl.col("venta_id"),
    pl.col("producto").alias("artículo"),
    pl.col("precio").cast(pl.Float64),
    (pl.col("precio") * pl.col("unidades").fill_null(0)).alias("importe"),
    pl.lit("EUR").alias("moneda"),
)

filtradas = ventas.filter(
    (pl.col("categoria") == "papelería")
    & (pl.col("precio") >= 3)
    & (pl.col("unidades").fill_null(0) > 2)
)

print(resumen)
print(filtradas)

## 3. `with_columns`: limpiar y enriquecer

`with_columns` conserva las columnas existentes y añade o reemplaza las expresiones indicadas. Aquí normalizamos el tipo, completamos el nulo y calculamos el importe y una etiqueta.

In [ ]:
ventas_limpias = ventas.with_columns(
    pl.col("unidades").fill_null(0).cast(pl.Int64),
    (pl.col("precio") * pl.col("unidades").fill_null(0)).round(2).alias("importe"),
    pl.when(pl.col("precio") >= 20)
      .then(pl.lit("alto"))
      .otherwise(pl.lit("normal"))
      .alias("segmento_precio"),
)
print(ventas_limpias)

## 4. Nulos, duplicados y ordenación

Podemos inspeccionar los nulos, eliminar duplicados con `unique` y ordenar con `sort`. La segunda fila de `venta_id=6` es deliberadamente repetida.

In [ ]:
print("Nulos por columna:")
print(ventas.null_count())

ventas_sin_duplicados = ventas_limpias.unique(subset=["venta_id"], keep="first", maintain_order=True)
ordenadas = ventas_sin_duplicados.sort(["categoria", "importe"], descending=[False, True])
print(ordenadas)

## 5. Agrupar y agregar

`group_by().agg()` permite calcular varias métricas por categoría. Las agregaciones ignoran los nulos en operaciones como `sum` y `mean`; antes hemos completado las unidades para que el importe sea explícito.

In [ ]:
por_categoria = (
    ventas_sin_duplicados
    .group_by("categoria")
    .agg(
        ventas=pl.len(),
        unidades_totales=pl.col("unidades").sum(),
        facturacion=pl.col("importe").sum().round(2),
        precio_medio=pl.col("precio").mean().round(2),
        productos=pl.col("producto").unique().sort(),
    )
    .sort("facturacion", descending=True)
)
print(por_categoria)

## 6. Join con validación de cardinalidad

Unimos las ventas con un catálogo de clientes. `validate="m:1"` documenta que muchas ventas pueden corresponder a un único cliente y hace fallar el join si la tabla de clientes contiene claves repetidas. Esta opción está disponible en versiones actuales de Polars.

In [ ]:
clientes = pl.DataFrame({
    "cliente_id": [101, 102, 103, 104],
    "cliente": ["Alba", "Bruno", "Carla", "Diego"],
    "localidad": ["Cádiz", "San Fernando", "Cádiz", "Chiclana"],
})

ventas_clientes = ventas_sin_duplicados.join(
    clientes,
    on="cliente_id",
    how="left",
    validate="m:1",
)
print(ventas_clientes.select("venta_id", "cliente", "localidad", "producto", "importe"))

## 7. Flujo lazy: plan y ejecución

Con `.lazy()` las operaciones se describen sin ejecutarse inmediatamente. `explain()` permite inspeccionar el plan optimizado y `collect()` materializa el resultado al final.

In [ ]:
consulta_lazy = (
    ventas_sin_duplicados.lazy()
    .filter(pl.col("importe") > 10)
    .group_by("cliente_id")
    .agg(
        importe_total=pl.col("importe").sum().round(2),
        numero_ventas=pl.len(),
    )
    .sort("importe_total", descending=True)
)

print("Plan optimizado:")
print(consulta_lazy.explain())
resultado_lazy = consulta_lazy.collect()
print("Resultado:")
print(resultado_lazy)

## Ideas clave

- Las expresiones Polars son composables y permiten mantener transformaciones declarativas.
- `with_columns` enriquece sin perder las columnas originales.
- `group_by().agg()` resume datos y `join(..., validate="m:1")` ayuda a detectar relaciones incorrectas.
- El modo lazy permite optimizar el plan antes de ejecutar `collect()`.